In [1]:
# vdb_strategy_name="GROBID_HF_Paragraph"
db_strategy_name="grobid"

In [2]:
from episcope.db.in_memory_academic_db import InMemoryAcademicDB
db = InMemoryAcademicDB("full_WP2_db.json")

INFO:episcope.db.in_memory_academic_db:Loading backup from full_WP2_db.json


In [3]:
from episcope.vectordb.qdrant import QdrantDB
vdb = QdrantDB(collection="wp2_HF", url="http://localhost:6334")

INFO:httpx:HTTP Request: GET http://localhost:6334 "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: GET http://localhost:6334/collections/wp2_HF "HTTP/1.1 200 OK"


In [4]:
all_papers = db.list_papers(strategy_name=db_strategy_name)

In [5]:
from episcope.rag.retrieval.semantic import SemanticRetriever
retriever = SemanticRetriever(vdb)

INFO:datasets:PyTorch version 2.6.0 available.
INFO:datasets:Polars version 1.34.0 available.
INFO:httpx:HTTP Request: POST http://localhost:6334/collections/wp2_HF/points/scroll "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST http://localhost:6334/collections/wp2_HF/points/scroll "HTTP/1.1 200 OK"
INFO:episcope.rag.retrieval.semantic:VectorDB is configured with embedding model: 'sentence-transformers/all-MiniLM-L6-v2'. Instantiating corresponding embedder for retrieval.
INFO:episcope.rag.embeddings.factory:Detected HuggingFace model 'sentence-transformers/all-MiniLM-L6-v2'. Creating HuggingFaceEmbedder.
INFO:sentence_transformers.SentenceTransformer:Load pretrained SentenceTransformer: sentence-transformers/all-MiniLM-L6-v2
INFO:sentence_transformers.SentenceTransformer:2 prompts are loaded, with the keys: ['query', 'text']


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

In [6]:
from episcope.rag.generation.llm_generator import LLMGenerator
generator = LLMGenerator(model="qwen2.5vl:3b") # "qwen2.5vl:3b"

In [7]:
from episcope.workflows import PaperClassifier
from episcope.workflows.classification import DataAccessibilityClassifierConfig, DataTypeClassifierConfig, PaperTypeClassifierConfig
classifier = PaperClassifier(
    retriever=retriever,
    generator=generator,
    academic_db=db,
    strategy_name=db_strategy_name,
    config=PaperTypeClassifierConfig()
)

In [8]:
import pandas as pd
list_results = []
for paper_id in all_papers[:1]:
    c_res = classifier.run(paper_id)
    result = {
    "paper_id": paper_id,
    "classification": c_res.classification.name,
    "class_probabilities": c_res.class_probabilities,
    "confidence": c_res.confidence,
    "evidence": c_res.evidence["reasoning"]
    }
    list_results.append(result)

df_results = pd.DataFrame(list_results)



Batches:   0%|          | 0/1 [00:00<?, ?it/s]

INFO:httpx:HTTP Request: POST http://localhost:6334/collections/wp2_HF/points/search "HTTP/1.1 200 OK"


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

INFO:httpx:HTTP Request: POST http://localhost:6334/collections/wp2_HF/points/search "HTTP/1.1 200 OK"


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

...

INFO:httpx:HTTP Request: POST http://localhost:6334/collections/wp2_HF/points/search "HTTP/1.1 200 OK"


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

INFO:httpx:HTTP Request: POST http://localhost:6334/collections/wp2_HF/points/search "HTTP/1.1 200 OK"


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

INFO:httpx:HTTP Request: POST http://localhost:6334/collections/wp2_HF/points/search "HTTP/1.1 200 OK"


In [ ]:
df_results = pd.DataFrame(list_results)
df_results

,paper_id,classification,class_probabilities,confidence,evidence
0,1976_Elveback_latent_period,LITERATURE_REVIEW,"{'A': 0.9, 'B': 0.1, 'C': 0.0}",0.9,The paper presents a stochastic simulation epi...


In [ ]:
GT_PAPER_TYPES = {
    "2020_He_infectious_period": "data",
    "2020_Lin_r0_basic_": "review",
    "2021_Ahammed_r0": "review",
    "2021_Zhu_infectious_period": "data",
    "2022_Du_k": "review",
    "2020_Xie_r0_basic_": "review",
    "2020_Izadi_r0_basic_": "review",
    "2020_Park_ifr": "review",
    "2020_Rai_serial_interval": "review",
    "2020_Yang_serial_interval": "data",
    "2021_Alene_incubation_period": "review",
    "2021_Ali_serial_interval": "review",
    "2021_Davies_r0": "data",
    "2021_Liu_and_Rocklöv_r0_basic_": "review",
    "2022_Águila-Mejía_infectious_period": "data",
    "2022_Garcia-Knight_infectious_period": "data",
    "2022_Guo_k": "review",
    "2022_Hart_latent_period": "data",
    "2022_Heiden_and_Buchholz_serial_interval": "data",
    "2022_Kremer_serial_interval": "data",
    "2022_Liu_r0_basic_": "review",
    "2022_Ryu__k": "data",
    "2022_Wu_incubation_period": "review", # PDF WAS WRONG (only supp was available)
    "2022_Zhao_k": "data",
    "2023_Galmiche_incubation_period": "data",
    "2023_Xu_incubation_period": "review",
    "2023_Yuan_cfr": "review",
    "2023_Zeng_serial_interval": "data",
    "2023_Zhang_and_Nishiura__ifr": "data",
    "2024_Ahmad__cfr": "review",
    "2024_Li__incubation_period": "data",
    "2024_Ward_ifr": "data"
}

In [ ]:
len(GT_PAPER_TYPES)

32

In [ ]:
from sklearn.metrics import balanced_accuracy_score

# Compute accuracy
correct = 0
total_count = 0
y_true = []
y_pred = []

for index, row in df_results.iterrows():
    paper_id = row["paper_id"]
    if paper_id in GT_PAPER_TYPES:
        predicted_class = row["classification"]
        gt_class = GT_PAPER_TYPES.get(paper_id, None)
        if gt_class == "data":
            gt_class = "DATA_ANALYSIS"
        elif gt_class == "review":
            gt_class = "LITERATURE_REVIEW"
        if gt_class == predicted_class:
            correct += 1
        total_count += 1
        y_true.append(gt_class)
        y_pred.append(predicted_class)

accuracy = correct / total_count if total_count > 0 else 0
balanced_accuracy = balanced_accuracy_score(y_true, y_pred) if total_count > 0 else 0

print(f"Accuracy: {accuracy*100:.2f}%")
print(f"Balanced Accuracy: {balanced_accuracy*100:.2f}%")

Accuracy: 0.00%
Balanced Accuracy: 0.00%


In [ ]:
# df_results.to_csv("paper_type_classification_results_qwen2.5vl_NO_SNIPPETS.csv", index=False)

In [ ]:
# c_res.confidence